In [ ]:

!pip -q install pgmpy

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

print("Libraries imported successfully!")



url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

columns = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs',
    'restecg', 'thalach', 'exang', 'oldpeak',
    'slope', 'ca', 'thal', 'target'
]

df = pd.read_csv(url, names=columns, na_values='?')

print("\nDataset downloaded successfully!")
print("\nFirst five records:")
print(df.head())

print("\nDataset shape:", df.shape)
print("\nDataset information:")
print(df.info())



df = df.apply(pd.to_numeric, errors='coerce')

df.dropna(inplace=True)



df['target'] = (df['target'] > 0).astype(int)



df['age'] = pd.cut(
    df['age'],
    bins=[0, 40, 50, 60, 100],
    labels=['Young', 'Middle', 'Senior', 'Elderly']
)

df['trestbps'] = pd.cut(
    df['trestbps'],
    bins=[0, 120, 140, 300],
    labels=['Normal', 'Elevated', 'High']
)

df['chol'] = pd.cut(
    df['chol'],
    bins=[0, 200, 240, 1000],
    labels=['Normal', 'Borderline', 'High']
)

df['thalach'] = pd.cut(
    df['thalach'],
    bins=[0, 100, 150, 250],
    labels=['Low', 'Medium', 'High']
)

df['oldpeak'] = pd.cut(
    df['oldpeak'],
    bins=[-1, 0, 2, 10],
    labels=['Low', 'Medium', 'High']
)


for col in df.columns:
    df[col] = df[col].astype('category').cat.codes

print("\nPreprocessing completed!")
print(df.head())



train_data, test_data = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['target']
)

print("\nTraining records:", len(train_data))
print("Testing records:", len(test_data))


model = DiscreteBayesianNetwork([
    ('age', 'heart_disease'),
    ('sex', 'heart_disease'),
    ('cp', 'heart_disease'),
    ('trestbps', 'heart_disease'),
    ('chol', 'heart_disease'),
    ('fbs', 'heart_disease'),
    ('restecg', 'heart_disease'),
    ('thalach', 'heart_disease'),
    ('exang', 'heart_disease'),
    ('oldpeak', 'heart_disease'),
    ('slope', 'heart_disease'),
    ('ca', 'heart_disease'),
    ('thal', 'heart_disease')
])

train_data = train_data.rename(
    columns={'target': 'heart_disease'}
)

test_data = test_data.rename(
    columns={'target': 'heart_disease'}
)



model.fit(
    train_data,
    estimator=BayesianEstimator,
    prior_type='BDeu',
    equivalent_sample_size=10
)

print("\nBayesian Network trained successfully!")


for cpd in model.get_cpds():
    print("\n", cpd)


inference = VariableElimination(model)



print("\n======================================")
print("HEART DISEASE PROBABILITY ESTIMATION")
print("======================================")


patient = {
    'age': 2,
    'sex': 1,
    'cp': 4,
    'trestbps': 1,
    'chol': 2,
    'fbs': 0,
    'restecg': 0,
    'thalach': 1,
    'exang': 1,
    'oldpeak': 2,
    'slope': 1,
    'ca': 1,
    'thal': 2
}

result = inference.query(
    variables=['heart_disease'],
    evidence=patient,
    show_progress=False
)

probability = result.values[1] * 100

print("\nPatient Evidence:")
print(patient)

print("\nProbability Distribution:")
print(result)

print(
    "\nEstimated probability of heart disease:",
    round(probability, 2), "%"
)



predictions = []
probabilities = []

for _, row in test_data.iterrows():

    evidence = row.drop('heart_disease').to_dict()

    result = inference.query(
        variables=['heart_disease'],
        evidence=evidence,
        show_progress=False
    )

    probabilities.append(result.values[1])

    predictions.append(
        int(np.argmax(result.values))
    )

actual = test_data['heart_disease'].astype(int)



accuracy = accuracy_score(actual, predictions)

print("\n======================================")
print("MODEL EVALUATION")
print("======================================")

print("Accuracy:", round(accuracy * 100, 2), "%")

print("\nClassification Report:")
print(
    classification_report(
        actual,
        predictions,
        zero_division=0
    )
)



plt.figure(figsize=(8, 5))

sns.histplot(
    np.array(probabilities) * 100,
    bins=15,
    kde=True
)

plt.title("Bayesian Network - Event Probability Distribution")
plt.xlabel("Probability of Heart Disease (%)")
plt.ylabel("Number of Patients")

plt.show()


plt.figure(figsize=(6, 5))

sns.heatmap(
    pd.crosstab(
        actual,
        predictions,
        rownames=['Actual'],
        colnames=['Predicted']
    ),
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title("Bayesian Network - Confusion Matrix")
plt.show()

print("\n======================================")
print("FINAL RESULT")
print("======================================")

print("Dataset: UCI Heart Disease Dataset")
print("Algorithm: Bayesian Network")
print("Learning: Bayesian Parameter Estimation")
print("Inference: Variable Elimination")
print("Event: Heart Disease")

print(
    "Estimated event probability:",
    round(probability, 2), "%"
)

print(
    "Test Accuracy:",
    round(accuracy * 100, 2), "%"
)

print("\nProgram executed successfully!")


Libraries imported successfully!

Dataset downloaded successfully!

First five records:
    age  sex   cp  trestbps   chol  fbs  restecg  thalach  exang  oldpeak  \
0  63.0  1.0  1.0     145.0  233.0  1.0      2.0    150.0    0.0      2.3   
1  67.0  1.0  4.0     160.0  286.0  0.0      2.0    108.0    1.0      1.5   
2  67.0  1.0  4.0     120.0  229.0  0.0      2.0    129.0    1.0      2.6   
3  37.0  1.0  3.0     130.0  250.0  0.0      0.0    187.0    0.0      3.5   
4  41.0  0.0  2.0     130.0  204.0  0.0      2.0    172.0    0.0      1.4   

   slope   ca  thal  target  
0    3.0  0.0   6.0       0  
1    2.0  3.0   3.0       2  
2    2.0  2.0   7.0       1  
3    3.0  0.0   3.0       0  
4    1.0  0.0   3.0       0  

Dataset shape: (303, 14)

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    float64